# ✅ **Score Provider Integration and Validation Notebook**

This notebook explicitly validates the integration and correctness of the **Score Provider**, including database initialization, loading extensions, computing scores, checking logs, and explicitly testing circular dependencies.

In [1]:
from pathlib import Path
import os, sys

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_scores import seed_score_providers
from app.db.seeders.seed_tools import seed_tool_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_tool_providers(session)
    seed_score_providers(session)


Working directory is now: C:\Repos\codecritic
Seeded AgentPrompt GUID: c9a76e0c-4ba6-459f-8449-84735ad79156
Seeded SystemPrompt GUID: 1b0c96a2-6226-4c18-ba7b-dd6bcee144d8
Seeded tool configurations successfully.
Seeded score providers successfully.


# ✅ **Build up score provider**

In [2]:
from sqlalchemy import select
from sqlalchemy.orm import Session
from app.db.models import ToolProviderConfig, ScoreProviderConfig
from app.factories.score_provider_factory import ScoreProviderFactory
from app.factories.tool_provider_factory import ToolProviderFactory

# 🔍 Fetch Black Formatter record from DB
with Session(bind=engine) as session:
    black_tool = session.execute(
        select(ToolProviderConfig).where(ToolProviderConfig.name == "Black Formatter")
    ).scalar_one()
    black_tool_id = black_tool.id

print(f"🆔 Black Formatter ID: {black_tool_id} 😄")

# 🛠️ Instantiate the ToolProvider using factory
black_instance = ToolProviderFactory.create(black_tool_id)
print(f"✅ ToolProvider instantiated: {black_instance.__class__.__name__}")

# 🔍 Fetch Lint Score Provider ID from DB
with Session(bind=engine) as session:
    lint_score_provider_record = session.execute(
        select(ScoreProviderConfig).where(ScoreProviderConfig.name == "Lint Score Provider")
    ).scalar_one()
    lint_score_provider_id = lint_score_provider_record.id

# 🔌 Instantiate ScoreProvider with tool only
score_provider = ScoreProviderFactory.create(
    lint_score_provider_id,
    tool_providers={"black": black_instance},
    context_provider=None,
)

print(f"✅ ScoreProvider instantiated: {score_provider.__class__.__name__}")

🆔 Black Formatter ID: 1 😄
✅ ToolProvider instantiated: BlackToolProvider
✅ ScoreProvider instantiated: LintScoreProvider


## 🔖 **Step 3: Execute Scoring Operation**

Perform a scoring operation on sample Python code and explicitly verify that scoring completes without errors.

In [3]:
from app.db.schemas import ScoreContext  # ensure correct import based on your current setup
from pathlib import Path

test_context = ScoreContext(
    experiment_id="test_exp_001",
    round=1,
    file_path=Path("tests/example.py"),
    source_code="def example():\n    pass\n"
)

scores = score_provider.score(test_context)
print("✅ Computed Scores:", scores)


✅ Computed Scores: {'formatting_compliance': 100.0, 'lint_error_count': 0, 'type_error_count': 0, 'average_complexity': 0.0, 'maintainability_index': 0.0}


## 🔖 **Step 4: Validate Score Logs**

Explicitly query the database to verify that score logs are correctly persisted.

In [4]:
from sqlalchemy import create_engine, text
from app.db.connection import DB_PATH

engine = create_engine(f"sqlite:///{DB_PATH}")

with engine.connect() as conn:
    result = conn.execute(
        text("SELECT experiment_id, round, score_provider_id, scores, timestamp FROM score_provider_log WHERE experiment_id='test_exp_001'")
    )
    score_logs = result.fetchall()

assert len(score_logs) > 0, "❌ No score logs found for test_exp_001!"

print("✅ Verified Score Logs:")
for log in score_logs:
    print(log)


✅ Verified Score Logs:
('test_exp_001', 1, 1, '{"formatting_compliance": 100.0, "lint_error_count": 0, "type_error_count": 0, "average_complexity": 0.0, "maintainability_index": 0.0}', '2025-05-24T17:29:10.961624+00:00')


## 🔖 **Step 5: Circular Dependency Check (Score Provider ↔ Tool Provider)**

This step explicitly verifies that circular dependencies between score providers and tool providers are resolved correctly during instantiation.

In [5]:
# 🔖 Step 5: Circular Dependency Check (Score Provider ↔ Tool Provider)

# Explicitly verify that the score provider and tool provider reference each other correctly
assert hasattr(score_provider, 'tool_providers'), "❌ ScoreProvider missing 'tool_providers' attribute!"
assert 'black' in score_provider.tool_providers, "❌ ToolProvider 'black' not set in ScoreProvider!"

black_tool_provider = score_provider.tool_providers['black']

assert hasattr(black_tool_provider, 'score_provider'), "❌ ToolProvider missing 'score_provider' attribute!"
assert black_tool_provider.score_provider is score_provider, "❌ Circular reference not correctly resolved!"

print("✅ Circular dependency explicitly verified between Score Provider and Tool Provider.")


✅ Circular dependency explicitly verified between Score Provider and Tool Provider.


## 🔖 **Step 6: Instantiate Score Provider Without Tool Providers**

Explicitly instantiate a Score Provider without passing any Tool Providers, ensuring the provider handles optional dependencies correctly.

In [7]:
# 🔖 Step 6: Instantiate Score Provider Without Tool Providers

from sqlalchemy.orm import Session
from sqlalchemy import select
from app.db.models import ScoreProviderConfig  # ✅ fix here

with Session(bind=engine) as session:
    lint_score_provider_record = session.execute(
        select(ScoreProviderConfig).where(ScoreProviderConfig.name == "Lint Score Provider")
    ).scalar_one()
    lint_score_provider_id = lint_score_provider_record.id

# Instantiate explicitly without any Tool Providers
score_provider_no_tools = ScoreProviderFactory.create(
    lint_score_provider_id,
    tool_providers=None,
    context_provider=None,
)

# Verify instantiation explicitly handles missing Tool Providers
assert score_provider_no_tools is not None, "❌ ScoreProvider instantiation without tool providers failed!"
assert score_provider_no_tools.tool_providers == {}, "❌ Tool providers should default to empty dict!"

print(f"✅ ScoreProvider instantiated without Tool Providers: {score_provider_no_tools.__class__.__name__}")


✅ ScoreProvider instantiated without Tool Providers: LintScoreProvider


## 📌 **Notebook Validation Summary**

✅ **Score Provider Integration Verified**:

- Database initialization and seeding explicitly confirmed.
- Extension loaded and instantiated correctly from database.
- Scoring operations executed successfully.
- Score logs persisted correctly in database.
- Circular dependency explicitly handled and tested.
- Optional tool-provider dependency explicitly verified.

🎉 **All explicit validations completed successfully. Your Score Provider integration is robust and ready for next phases!**